# CSCI 4150 Starter Notebook: When Similarity Misleads

Run cells in order. This notebook is designed to make semantic-search failure, ambiguity, context loss, and score over-trust inspectable. Save outputs and short answers for your submissions.


In [ ]:
# Colab setup. Run this once if sentence-transformers is not already installed.
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    %pip -q install sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# The corpus contains deliberate near-misses: ambiguity, negation, context/date, and domain distinctions.
documents = [
    "Machine Learning requires Data Structures and Linear Algebra.",
    "Machine Learning does not require Operating Systems.",
    "Students may request an extension before the deadline.",
    "Extensions are not normally granted after the deadline.",
    "The 2023 course catalog lists Data Structures as a prerequisite.",
    "The 2026 course catalog lists Data Structures and Linear Algebra as prerequisites.",
    "Python is used for data science and machine learning.",
    "The python is a large nonvenomous snake.",
    "Java is a programming language commonly used for backend systems.",
    "Java is an Indonesian island.",
    "A machine-learning model maps inputs to predictions.",
    "A scale model represents a physical object at reduced size.",
    "Goggles are required when working with corrosive chemicals.",
    "Goggles are not required for ordinary computer-lab exercises.",
    "The AI ethics seminar examines bias, privacy, labor, and governance.",
    "Office hours are available after class and by appointment.",
]

model = SentenceTransformer("all-MiniLM-L6-v2")
doc_vectors = model.encode(documents)
print(f"Loaded {len(documents)} documents.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded 16 documents.


In [ ]:
def search(query, k=3):
    """Return the top-k documents and cosine-similarity scores for a query."""
    query_vector = model.encode([query])
    scores = cosine_similarity(query_vector, doc_vectors)
    ranked = scores[0].argsort()[::-1][:k]
    return [(documents[i], float(scores[0][i])) for i in ranked]


def show_results(query, k=3):
    print(f"\nQUERY: {query}")
    print("-" * 80)
    for rank, (doc, score) in enumerate(search(query, k=k), start=1):
        print(f"{rank}. {score:.3f} | {doc}")

show_results("Can I take Machine Learning without Linear Algebra?", k=3)



QUERY: Can I take Machine Learning without Linear Algebra?
--------------------------------------------------------------------------------
1. 0.738 | Machine Learning requires Data Structures and Linear Algebra.
2. 0.538 | Machine Learning does not require Operating Systems.
3. 0.456 | The 2026 course catalog lists Data Structures and Linear Algebra as prerequisites.


## Failure-label key

- **A — Ambiguity failure:** the query has multiple plausible meanings, and retrieval chooses or mixes the wrong sense.
- **N — Negation/constraint failure:** the result is related but ignores or reverses *not*, *without*, *except*, *before*, *after*, or *only*.
- **C — Context-loss failure:** important context such as date, user role, course level, jurisdiction, or prerequisite is missing.
- **S — Similarity is not correctness:** a high score indicates closeness in the embedding space, not factual correctness or answer correctness.
- **D — Domain mismatch:** general semantic similarity misses a domain-specific or technical distinction.
- **R — Representation/data limitation:** the result suggests a representational gap worth investigating; it does not prove the source of that gap by itself.


In [ ]:
test_queries = [
    ("Can I take Machine Learning without Linear Algebra?", "negation/constraint"),
    ("What are the current prerequisites for Machine Learning?", "context/date"),
    ("Tell me about Python.", "ambiguity"),
    ("How dangerous is a python?", "ambiguity"),
    ("Do I need goggles in the computer lab?", "negation/context"),
    ("What kind of model predicts outcomes?", "domain ambiguity"),
    ("Can I get an extension after the deadline?", "exception/constraint"),
]

for query, expected_issue in test_queries:
    print(f"\nExpected issue to inspect: {expected_issue}")
    show_results(query, k=3)



Expected issue to inspect: negation/constraint

QUERY: Can I take Machine Learning without Linear Algebra?
--------------------------------------------------------------------------------
1. 0.738 | Machine Learning requires Data Structures and Linear Algebra.
2. 0.538 | Machine Learning does not require Operating Systems.
3. 0.456 | The 2026 course catalog lists Data Structures and Linear Algebra as prerequisites.

Expected issue to inspect: context/date

QUERY: What are the current prerequisites for Machine Learning?
--------------------------------------------------------------------------------
1. 0.615 | Machine Learning requires Data Structures and Linear Algebra.
2. 0.488 | The 2026 course catalog lists Data Structures and Linear Algebra as prerequisites.
3. 0.467 | Machine Learning does not require Operating Systems.

Expected issue to inspect: ambiguity

QUERY: Tell me about Python.
--------------------------------------------------------------------------------
1. 0.797 | Py

In [ ]:
# Top-k experiment: choose at least two queries and compare k=1, k=3, and k=5.
queries_to_compare = [
    "Can I take Machine Learning without Linear Algebra?",
    "Tell me about Python.",
]

for query in queries_to_compare:
    for k in [1, 3, 5]:
        show_results(query, k=k)



QUERY: Can I take Machine Learning without Linear Algebra?
--------------------------------------------------------------------------------
1. 0.738 | Machine Learning requires Data Structures and Linear Algebra.

QUERY: Can I take Machine Learning without Linear Algebra?
--------------------------------------------------------------------------------
1. 0.738 | Machine Learning requires Data Structures and Linear Algebra.
2. 0.538 | Machine Learning does not require Operating Systems.
3. 0.456 | The 2026 course catalog lists Data Structures and Linear Algebra as prerequisites.

QUERY: Can I take Machine Learning without Linear Algebra?
--------------------------------------------------------------------------------
1. 0.738 | Machine Learning requires Data Structures and Linear Algebra.
2. 0.538 | Machine Learning does not require Operating Systems.
3. 0.456 | The 2026 course catalog lists Data Structures and Linear Algebra as prerequisites.
4. 0.384 | A machine-learning model maps i

In [ ]:
# Create a blank evidence table for your project dossier. Fill this in during studio.
columns = [
    "Query", "Top result", "Score", "Relevant?", "Failure label", "Evidence", "Proposed fix"
]
analysis_table = pd.DataFrame(columns=columns)
analysis_table


,Query,Top result,Score,Relevant?,Failure label,Evidence,Proposed fix


## Required artifact

Complete at least four cases:

1. one ambiguity case;
2. one negation or exception case;
3. one context, date, or domain case;
4. one adversarial query designed by another team.

For each case, explain what the similarity score captured and what it ignored. Then propose one semantic structure that could help: metadata filtering, Boolean/constraint processing, hybrid search, reranking, structured knowledge, or adding more context to the query/documents.


In [ ]:
# Optional: export the evidence table after you fill it in.
# analysis_table.to_csv("meeting08_semantic_search_error_analysis.csv", index=False)
